# Generative Image Denoising -- Automated Colab Training Notebook

This notebook trains and evaluates the multi-architecture generative image denoising pipeline directly on a Google Colab GPU.

### **One-Click "Run All" Workflow:**
1. Switch runtime to GPU (**Runtime -> Change runtime type -> T4 GPU**).
2. Click **Runtime -> Run all**.

Everything is fully automated:
- Mounts Google Drive to cache datasets and save results persistently.
- Clones the repository directly from GitHub into fast local storage.
- Installs requirements and verifies GPU & layers via smoke test.
- Downloads DIV2K + BSDS500 datasets **once** to your Drive cache.
- Trains the selected architecture (`nafnet_unet` by default).
- Evaluates PSNR, SSIM, DISTS, and FLOPs on the validation set.
- Packages submission-ready weights (`model_weights.pt` + `inference.py`).
- Saves full results to Drive and downloads the `<ARCH>_results.zip` package.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Configuration

Set `ARCH` to the architecture you want to train this run, and confirm
the Drive paths below. `PROJECT_ZIP_IN_DRIVE` should point at the project
zip you uploaded to Drive in the one-time setup step.


In [ ]:
import os

# ---- Choose which architecture to train this run ----
ARCH = "nafnet_unet"  # options: "nafnet_unet", "pix2pix_gan", "restormer_lite", "tiny_ddpm_sr3"

# ---- Persistent Google Drive paths ----
DRIVE_ROOT = "/content/drive/MyDrive/denoising_competition"
DATA_ROOT = f"{DRIVE_ROOT}/data"                      # dataset cache lives here PERMANENTLY
RESULTS_ROOT_DRIVE = f"{DRIVE_ROOT}/results"          # a persistent copy of results also lives here

# ---- Local working directory inside Colab VM (fast SSD) ----
LOCAL_PROJECT_DIR = "/content/project"

os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(RESULTS_ROOT_DRIVE, exist_ok=True)

print(f"Training architecture: {ARCH}")
print(f"Dataset cache (persistent): {DATA_ROOT}")
print(f"Results backup (persistent): {RESULTS_ROOT_DRIVE}")


## 3. Clone the project code from GitHub

Clones the repository directly from GitHub into `/content/project`.
If the directory already exists, it pulls the latest updates.


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/Sagnik120/generative-image-denoising.git"

if not Path(LOCAL_PROJECT_DIR).exists():
    print(f"Cloning {REPO_URL} into {LOCAL_PROJECT_DIR} ...")
    !git clone {REPO_URL} {LOCAL_PROJECT_DIR}
else:
    print(f"{LOCAL_PROJECT_DIR} already exists. Pulling latest commits...")
    %cd {LOCAL_PROJECT_DIR}
    !git pull

%cd {LOCAL_PROJECT_DIR}
print("Working directory:", os.getcwd())


## 4. Install dependencies

Colab already ships with `torch`, so this mainly adds the metric/FLOPs packages.

In [ ]:
!pip install -q -r requirements.txt


## 5. Verify GPU is active
If this prints `cuda`, you're good. If it prints `cpu`, go to
`Runtime -> Change runtime type -> GPU` and re-run from the top.


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 6. Quick sanity check (optional but recommended)

Runs a fast smoke test of every architecture's forward/backward pass on
tiny dummy data (a few seconds), to catch any environment issues before
committing to a long real training run.


In [ ]:
!python scripts/verify_all.py


## 7. Download / verify the dataset (ONE-TIME, cached in Drive)

This downloads DIV2K (train + valid) and BSDS500 into `DATA_ROOT` on your
Drive. If they're already there from a previous run, this cell does
**nothing** and returns immediately -- safe to re-run every session.

First run: ~10-20 minutes depending on Colab's network speed (~3.7GB total).
Every run after that: a few seconds (just checks the files exist).


In [ ]:
import sys
sys.path.insert(0, LOCAL_PROJECT_DIR)
from src.common.dataset import ensure_datasets

dataset_paths = ensure_datasets(DATA_ROOT, download_bsds=True)
print("\nDataset folders ready:")
for k, v in dataset_paths.items():
    print(f"  {k}: {v}")


### Optional: add low-light / medical domain images

The held-out evaluation set spans natural photography, low-light imagery,
and medical imaging (X-ray, MRI). The core pipeline above already covers
natural images broadly (DIV2K + BSDS500). To also add a slice of
low-light or medical images (recommended, see `docs/dataset_notes.md` for
where to source license-appropriate ones):

1. Upload your extra images to Drive at:
   `DATA_ROOT/extra/<domain_name>/*.png` (e.g. `.../extra/lowlight/*.png`,
   `.../extra/chest_xray/*.png`)
2. Re-run the cell above -- `ensure_datasets` automatically detects and
   includes any folders under `data/extra/`.

This step is optional; the pipeline works fully without it.


## 8. Train

This runs `scripts/train.py` for the architecture chosen in the
Configuration cell. Training writes checkpoints, logs, loss curves, and
visualizations into `results/<ARCH>/` on the LOCAL Colab disk as it goes
(fast), and you will sync everything to Drive as a zip at the end.

Adjust `EPOCHS` / `BATCH_SIZE` here to override that architecture's
config.yaml if you want a quicker test run first (e.g. `EPOCHS=2`) before
committing to a full run.


In [ ]:
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()

EPOCHS = None       # e.g. 60; leave as None to use the architecture's own config.yaml value
BATCH_SIZE = 8      # 8 fits comfortably in Colab T4 15GB VRAM (16 causes CUDA OOM on T4)

cmd = f"python scripts/train.py --arch {ARCH} --data_root '{DATA_ROOT}'"
if EPOCHS is not None:
    cmd += f" --epochs {EPOCHS}"
if BATCH_SIZE is not None:
    cmd += f" --batch_size {BATCH_SIZE}"

print("Running:", cmd)
!{cmd}


## 9. Final evaluation + packaging

Runs a full validation-set evaluation (PSNR / SSIM / DISTS / FLOPs) with
the best checkpoint, then zips the entire `results/<ARCH>/` folder
(checkpoints, loss_curves, metrics, visualizations, logs) preserving the
exact same folder structure as your local project's `results/` folder.


In [ ]:
!python scripts/evaluate.py --arch {ARCH} --data_root '{DATA_ROOT}'


In [ ]:
# Export a clean, self-contained submission package
# (model_weights.pt + standalone inference.py + README.md)
!python scripts/export_inference.py --arch {ARCH}


## 10. Sync results to Drive (persistent) + download locally

Two things happen here:
1. The `results/<ARCH>/` folder is copied into your Drive
   (`RESULTS_ROOT_DRIVE`) so it survives even if you don't download it
   right now.
2. A zip of `results/<ARCH>/` is created and offered as a direct
   browser download -- unzip it straight into your LOCAL project's
   `results/` folder (it already matches that folder's internal
   structure, e.g. `nafnet_unet/checkpoints/...`,
   `nafnet_unet/loss_curves/...`, etc.).


In [ ]:
import shutil
from pathlib import Path
from google.colab import files

# 1) Persist a copy of results into Drive
drive_dest = Path(RESULTS_ROOT_DRIVE) / ARCH
if drive_dest.exists():
    shutil.rmtree(drive_dest)
shutil.copytree(Path("results") / ARCH, drive_dest)
print(f"[Persistent Backup] Copied results/{ARCH} -> {drive_dest}")

# 2) Package results into <ARCH>_results.zip
zip_base = f"/content/{ARCH}_results"
zip_path = shutil.make_archive(zip_base, "zip", root_dir="results", base_dir=ARCH)
print(f"[Packaged Zip] Created: {zip_path}")

# 3) Trigger browser download
try:
    files.download(zip_path)
    print("Download prompt triggered.")
except Exception as e:
    print(f"Note: browser auto-download skipped ({e}). Your results are safely backed up in Drive at {drive_dest}.")


## 11. (After training multiple architectures) Compare them all

Once you've run this notebook once per architecture (changing `ARCH` in
step 2 each time) and evaluated each with step 9, run this cell to get a
single side-by-side comparison table + bar chart across all four
competition axes (FLOPs, PSNR, SSIM, DISTS).

Tip: since each run only evaluates the architecture chosen in step 2, if
you want to compare all four in one notebook session, either re-run
steps 2, 8, and 9 with a different `ARCH` each time (results accumulate
under separate `results/<arch>/` folders and are not overwritten), or
pull all four already-evaluated `results/<arch>/` folders back down from
Drive into `results/` before running this cell.


In [ ]:
!python scripts/compare_architectures.py


---
## Notes on how this notebook avoids wasted work

- **Dataset**: `ensure_datasets()` (step 7) checks if the expected image
  files already exist under `DATA_ROOT` (your Drive) before downloading
  anything. A Colab disconnect/restart never costs you another 3.7GB
  download -- just re-run steps 1-3 and 7, and it will skip straight to
  training.
- **Results**: step 10 copies results into Drive AND gives you a direct
  zip download, so you have a persistent copy in two places even if the
  Colab VM is recycled before you remember to download anything.
- **Code**: only the project *code* is unzipped fresh into `/content`
  (fast local disk) each session; the large, persistent things (data,
  results) always live in Drive and are referenced by path.
